### 1. PROFILAGE ET MODÉLISATION

### 1. Analyser un relevé

In [ ]:
import json
from pathlib import Path
import pandas as pd

# Chemin correct vers ton fichier
path = Path("releves/chronovet/2026-04-01_1919/products.jsonl")

# print("Chemin utilisé :", path.resolve())
# print("Existe :", path.exists())

rows = [json.loads(line) for line in path.open(encoding="utf-8")]
df = pd.DataFrame(rows)

# 1) basic counts
n_rows = len(df)    #1631 produits
n_ean_missing = df["ean"].isna().sum() + (df["ean"] == "").sum()    #le nb des valeurs manquantes ou vides 

# 2) ean duplicates
dup_ean = df["ean"].value_counts()   #count the frequency of unique values
n_ean_duplicates = (dup_ean > 1).sum()

# 3) columns
columns = df.columns.tolist()

n_rows, n_ean_missing, n_ean_duplicates, columns


(1631,
 np.int64(0),
 np.int64(0),
 ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'price',
  'currency',
  'in_stock',
  'image_url',
  'extra'])

### 2. Analyser toutes les dates d’un site (chronovet)

In [19]:
def get_columns(path):
    rows = [json.loads(line) for line in Path(path).open(encoding="utf-8")]
    return pd.DataFrame(rows).columns.tolist()

base = Path("releves/chronovet")

columns_by_date = {}

for folder in base.iterdir():
    file = folder / "products.jsonl"
    if file.exists():                                       #sécurité
        columns_by_date[folder.name] = get_columns(file)

columns_by_date

{'2026-04-01_1919': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'price',
  'currency',
  'in_stock',
  'image_url',
  'extra'],
 '2026-07-12_1150': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'description_short',
  'species',
  'price',
  'price_was',
  'currency',
  'in_stock',
  'image_url',
  'extra',
  'conditioning',
  'weight_kg'],
 '2026-07-12_1242': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'description_short',
  'species',
  'price',
  'price_was',
  'currency',
  'in_stock',
  'image_url',
  'extra',
  'conditioning',
  'weight_kg'],
 '2026-07-18_1334': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'description_short',
  'species',
  'price',
  'price_was',
  'currency',
  'in_stock',
  'image_url',
  'extra',
  'conditioning',
  'weight_kg']}

### 3. Analyser tous les sites

In [13]:
def get_columns(path):
    rows = [json.loads(line) for line in Path(path).open(encoding="utf-8")]
    return pd.DataFrame(rows).columns.tolist()

root = Path("releves")

columns_all_sites = {}

for site in root.iterdir():
    if site.is_dir():
        for folder in site.iterdir():
            file = folder / "products.jsonl"
            if file.exists():
                columns_all_sites[f"{site.name}/{folder.name}"] = get_columns(file)

columns_all_sites

{'animalis/2026-04-01_1919': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'description_short',
  'variant_name',
  'pack_size',
  'price',
  'currency',
  'in_stock',
  'image_url',
  'extra'],
 'animalis/2026-07-12_1150': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'description_short',
  'variant_name',
  'pack_size',
  'price',
  'currency',
  'in_stock',
  'image_url',
  'extra'],
 'animalis/2026-07-12_1242': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'description_short',
  'variant_name',
  'pack_size',
  'price',
  'currency',
  'in_stock',
  'image_url',
  'extra'],
 'animalis/2026-07-18_1334': ['site',
  'url',
  'scraped_at',
  'ean',
  'sku',
  'site_id',
  'name',
  'brand',
  'category',
  'description_short',
  'variant_name',
  'pack_size',
  'price',
  'currency',
  'in_stock',
  'image_url',
  'extra

### 4.  les valeurs non nulles pour tous les sites  

In [ ]:
import os

#releves_folder = "releves"
releves_folder = os.path.join(os.path.dirname(__file__), "releves")  #dossier où se trouve le script Python

frames = []
for site in os.listdir(releves_folder):
    site_folder = os.path.join(releves_folder, site)
    if not os.path.isdir(site_folder):
        continue
    for run in os.listdir(site_folder):
        path = os.path.join(site_folder, run, "products.jsonl")
        if os.path.isfile(path):
            frames.append(pd.read_json(path, lines=True))

df_all = pd.concat(frames, ignore_index=True)   #concaténer des objets dans df
df_all.count()

site                    388784
url                     388784
scraped_at              388784
ean                     360862
sku                     369412
site_id                 214606
name                    388784
brand                   357725
category                210172
description_short       190378
variant_name            168509
pack_size                71205
price                   369192
currency                388784
in_stock                363874
image_url               365750
extra                   388784
price_was                24862
price_per_unit           47035
price_per_unit_label     93659
stock_text              100248
species                   8167
conditioning              1551
weight_kg                 4051
mpn                      14203
rating                    4070
review_count              4070
atc_code                     4
dtype: int64

### 5. Calculer les % EAN manquants globalement:

In [21]:
pct_ean_missing = (df_all["ean"].isna().sum() + (df_all["ean"] == "").sum()) / len(df_all) * 100
pct_ean_missing


np.float64(7.181879912753611)

### 6. Calculer les % doublons EAN globalement

In [16]:
dup = df_all["ean"].value_counts()
pct_ean_duplicates = (dup > 1).sum() / len(df_all) * 100
pct_ean_duplicates


np.float64(23.684359438660028)

### 7. Colonnes présentes dans tous les sites


In [18]:
df_all.columns.tolist()


['site',
 'url',
 'scraped_at',
 'ean',
 'sku',
 'site_id',
 'name',
 'brand',
 'category',
 'description_short',
 'variant_name',
 'pack_size',
 'price',
 'currency',
 'in_stock',
 'image_url',
 'extra',
 'price_was',
 'price_per_unit',
 'price_per_unit_label',
 'stock_text',
 'species',
 'conditioning',
 'weight_kg',
 'mpn',
 'rating',
 'review_count',
 'atc_code']

## 2. Chargement des relevés fournis

In [36]:
import os
import json
import pandas as pd
import psycopg2
import unicodedata

# -----------------------------
# Normalisation du nom produit
# -----------------------------
def normalize_name(name):
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = " ".join(name.split())
    return name

# -----------------------------
# Connexion PostgreSQL
# -----------------------------
conn = psycopg2.connect(
    dbname="vetprice",
    user="postgres",
    password="Mkilo1990",
    host="localhost",
    port=5432
)
cur = conn.cursor()

# -----------------------------
# Fonction : upsert produit
# -----------------------------
def upsert_product(row):
    """
    Retourne product_id (existant ou nouvellement créé)
    """

    # Convertir EAN en texte
    ean_raw = row.get("ean")
    if isinstance(ean_raw, float):  # pandas lit NaN comme float
        ean = None
    else:
        ean = str(ean_raw) if ean_raw not in [None, ""] else None

    # Convertir SKU en texte
    sku_raw = row.get("sku")
    if isinstance(sku_raw, float):  # pandas lit NaN comme float
        sku = None
    else:
        sku = str(sku_raw) if sku_raw not in [None, ""] else None

    site = row.get("site")
    name = row.get("name")
    name_norm = normalize_name(name)

    # 1. Si EAN présent → chercher par EAN
    if ean:
        cur.execute("SELECT product_id FROM product WHERE ean = %s", (ean,))
        res = cur.fetchone()
        if res:
            return res[0]

    # 2. Sinon → clé de repli (site, sku, name_normalized)
    cur.execute("""
        SELECT product_id FROM product
        WHERE site = %s AND name_normalized = %s AND (sku = %s OR sku IS NULL)
    """, (site, name_norm, sku))
    res = cur.fetchone()
    if res:
        return res[0]

    # 3. Insérer un nouveau produit
    cur.execute("""
        INSERT INTO product (ean, site, sku, name_normalized, name, brand, category,
                             description_short, variant_name, pack_size, species,
                             conditioning, weight_kg, mpn, atc_code, image_url, extra)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        RETURNING product_id
    """, (
        ean, site, sku, name_norm, name,
        row.get("brand"), row.get("category"), row.get("description_short"),
        row.get("variant_name"), row.get("pack_size"), row.get("species"),
        row.get("conditioning"), row.get("weight_kg"), row.get("mpn"),
        row.get("atc_code"), row.get("image_url"), json.dumps(row.get("extra"))
    ))
    return cur.fetchone()[0]

# -----------------------------
# Fonction : insérer un relevé
# -----------------------------
def insert_price_fact(row, product_id):
    cur.execute("""
        INSERT INTO price_fact (product_id, site, url, scraped_at,
                                price, price_was, currency, in_stock,
                                stock_text, price_per_unit, price_per_unit_label,
                                rating, review_count, extra)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT DO NOTHING
    """, (
        product_id,
        row.get("site"),
        row.get("url"),
        row.get("scraped_at"),
        row.get("price"),
        row.get("price_was"),
        row.get("currency"),
        row.get("in_stock"),
        row.get("stock_text"),
        row.get("price_per_unit"),
        row.get("price_per_unit_label"),
        row.get("rating"),
        row.get("review_count"),
        json.dumps(row.get("extra"))
    ))

# -----------------------------
# Chargement des relevés
# -----------------------------
def load_releve(path):
    print(f"Chargement : {path}")

    df = pd.read_json(path, lines=True)

    # Dédoublonnage interne du relevé
    df = df.drop_duplicates(subset=["ean", "sku", "name"], keep="first")

    for _, row in df.iterrows():
        row = row.to_dict()
        product_id = upsert_product(row)
        insert_price_fact(row, product_id)

    conn.commit()
    print("OK")

# -----------------------------
# Exécution : charger plusieurs dates
# -----------------------------
load_releve("releves/chronovet/2026-04-01_1919/products.jsonl")
load_releve("releves/chronovet/2026-07-12_1150/products.jsonl")
load_releve("releves/chronovet/2026-07-12_1242/products.jsonl")
load_releve("releves/chronovet/2026-07-18_1334/products.jsonl")

print("Chargement terminé.")


Chargement : releves/chronovet/2026-04-01_1919/products.jsonl
OK
Chargement : releves/chronovet/2026-07-12_1150/products.jsonl
OK
Chargement : releves/chronovet/2026-07-12_1242/products.jsonl
OK
Chargement : releves/chronovet/2026-07-18_1334/products.jsonl
OK
Chargement terminé.
